
# 19 库存修订与当时可见的数据

> **本章要解决的问题**：同一周的库存，第一次公布 100、后来修订 108。回测时你应该看到哪个数？用修订后的数，会偷看未来；用初值，又可能和当时系统不一致。

## 本章导读

第12章讲因子流水线，第14章讲对账。这一章讲**point-in-time（PIT）数据**：

- 数据在“当时”可见的版本；
- 修订/补发/延迟；
- merge_asof 对齐；
- 信号差异。

### 学习目标

- 能区分初值、修订值、最终值，并判断当时可见版本。
- 会用 merge_asof 按可见时间对齐。
- 能构造 PIT 面板，并保留版本/时间戳。
- 能比较初值/修订两套信号，估计未来函数的影响。

### 先修

第01章的四时刻；第12章的因子卡；会 pandas 的 merge_asof。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="19 库存修订与当时可见的数据" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>19 库存修订与当时可见的数据｜知识图</title>
<desc>本章主线：回测只能用“当时可见”的版本；初值、修订值、最终值要分开存、按到达时间对齐。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a28" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao28" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag28" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">19</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">库存修订与当时可见的数据</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：回测只能用“当时可见”的版本；初值、修订值、最终值要分开存、按到达时间对齐。</text>
<rect x="41" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="121" y="132.1" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">发布 / 修订 / 到达记</text>
<text x="121" y="150.6" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">录</text>
<text x="121" y="170.6" font-size="10.5" fill="#6E7B8B" text-anchor="middle">版本、时间戳、补发</text>
<rect x="51" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="68" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输入</text>
<rect x="227" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="307" y="132.1" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">merge_asof 按可见时</text>
<text x="307" y="150.6" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">间对齐</text>
<text x="307" y="170.6" font-size="10.5" fill="#6E7B8B" text-anchor="middle">到达 ≤ 决策时刻</text>
<rect x="237" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="254" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">处理</text>
<rect x="413" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="493" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">PIT 面板与信号</text>
<text x="493" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">保留版本与时间戳</text>
<rect x="423" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="440" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输出</text>
<rect x="599" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="679" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">初值 / 修订值对照</text>
<text x="679" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">估计未来函数的影响</text>
<rect x="609" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="626" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">检查</text>
<path d="M 201 149 L 227 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a28)"/>
<path d="M 387 149 L 413 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a28)"/>
<path d="M 573 149 L 599 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a28)"/>
<rect x="41" y="232" width="718" height="152" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<rect x="58" y="246" width="4" height="11" fill="#E07B39" rx="1"/>
<text x="70" y="256" font-size="11.5" fill="#1B3A5C" font-weight="bold">库存修订的可见性</text>
<text x="58" y="276" font-size="9.5" fill="#6E7B8B">同一周的库存：初值 100 → 修订 108 → 最终 110</text>
<path d="M 90 326 L 720 326" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag28)"/>
<circle cx="215" cy="326" r="5.5" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.8"/>
<text x="215" y="306" font-size="9" fill="#1B3A5C" text-anchor="middle">09:00 初值 100</text>
<circle cx="430" cy="326" r="5.5" fill="#FFFFFF" stroke="#E07B39" stroke-width="1.8"/>
<text x="430" y="306" font-size="9" fill="#6E7B8B" text-anchor="middle">09:10 修订 108</text>
<circle cx="645" cy="326" r="5.5" fill="#FFFFFF" stroke="#E07B39" stroke-width="1.8"/>
<text x="645" y="306" font-size="9" fill="#6E7B8B" text-anchor="middle">11:00 最终 110</text>
<rect x="90" y="318" width="250" height="8" fill="#E6EFF7" rx="3"/>
<text x="92" y="342" font-size="8.8" fill="#6E7B8B">当时可见（到达 ≤ 决策）</text>
<line x1="340" y1="286" x2="340" y2="334" stroke="#E07B39" stroke-width="1.4" stroke-linecap="round" stroke-dasharray="4 3"/>
<text x="336" y="298" font-size="8.8" fill="#E07B39" text-anchor="end" font-weight="bold">决策 09:06</text>
<text x="336" y="312" font-size="8.8" fill="#E07B39" text-anchor="end">只能看到 100</text>
<text x="540" y="360" font-size="8.8" fill="#6E7B8B">merge_asof：取 ≤ 时刻的最近记录</text>
</svg>

> **图**：本章主线：回测只能用“当时可见”的版本；初值、修订值、最终值要分开存、按到达时间对齐。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：数据工程、因子研究员、回测。
- **输入**：发布记录、修订版本、到达时间。
- **输出**：PIT 面板、当时可见信号、初值/修订对照。
- **在流程的哪一步**：数据准备与防泄漏。
- **失败模式**：用最终值、发布当到达、补发回填。
- **最小工作流**：发布/到达 → merge_asof → PIT 面板 → 信号 → 初值/修订对照。

---
## 1. 三种值

| 值 | 含义 | 能不能用于当时信号 |
|---|---|---|
| 初值 | 第一次公布 | 可以（如果当时已发布） |
| 修订值 | 后来修正 | 不可以（当时不知道） |
| 最终值 | 最后一次修订 | 不可以 |

**回测要用“当时可见”的版本，不是“现在数据库里的最终值”。**

---

## 2. merge_asof：按可见时间对齐

### 2.1 规则

对每个决策时刻，取“可见时间 ≤ 决策时刻”的最近一条发布记录。

### 2.2 代码



In [2]:

import pandas as pd

release = pd.DataFrame({
    "available": pd.to_datetime(["2026-11-06 16:03", "2026-11-09 09:08"]),
    "stock": [100, 108],
})
decisions = pd.DataFrame({
    "decision": pd.to_datetime(["2026-11-09 09:06", "2026-11-09 09:10"]),
})

print(pd.merge_asof(decisions, release, left_on="decision", right_on="available",
                    direction="backward"))


             decision           available  stock
0 2026-11-09 09:06:00 2026-11-06 16:03:00    100
1 2026-11-09 09:10:00 2026-11-09 09:08:00    108



**读数**：09:06 只能看到 100；09:10 才能看到 108。若把 108 用在 09:06，就是未来函数。

**常见错误**：

1. 用最终值回测。
2. 用“发布日期”而不是“实际可见时间”（网络延迟/补发）。
3. 不处理时区/交易日。

---

## 3. PIT 面板与信号差异

### 3.1 构造

每条记录保留：期间、发布时刻、到达时刻、值、版本。对每个决策时刻，取“到达 ≤ 决策”的最新版本。

### 3.2 代码：两个信号



In [3]:

previous, forecast, actual = 108, 100, 104
print("库存变化", actual - previous, "公告意外", actual - forecast)
print("越少越强信号（初值/修订）", previous - actual, forecast - actual)


库存变化 -4 公告意外 4
越少越强信号（初值/修订） 4 -4



**读数**：同一个 104，用 108 还是 100 做基准，信号方向/大小不同。PIT 面板要保留“当时用的基准”。

**常见错误**：

1. 只用最终值算变化。
2. 不记录版本。
3. 把“补发”当“按时发布”。

**停下来想一想**：为什么“用修订值回测”会虚高？

<details><summary>对照你的思路</summary>

修订值包含后来才知道的信息（比如库存被下修），当时交易者看不到；用它算信号等于偷看未来，回测会高估。

</details>

---

## 4. 案例：一次库存修订怎样改变信号

| 时点 | 库存 | 说明 |
|---|---:|---|
| 11-06 16:03 | 100 | 初值 |
| 11-09 09:08 | 108 | 修订 |
| 11-09 09:10 | 104 | 实际值/新一期 |

若信号是“库存越少越看多”：

- 用初值 100：$100-104=-4$，库存相对上升 → 看空；
- 用修订 108：$108-104=+4$，库存相对下降 → 看多。

**同一个 104，基准不同，信号方向相反。** 09:06 的决策只能看到 100；09:10 才能看到 108。

### 代码：两个信号



In [4]:

initial, revised, actual = 100, 108, 104
print("用初值的信号", initial - actual)
print("用修订的信号", revised - actual)
print("方向是否一致", (initial - actual) * (revised - actual) > 0)


用初值的信号 -4
用修订的信号 4
方向是否一致 False



**读数**：两个信号符号相反；若回测在 09:06 用了 108，就会把“未来修订”当成当时信息。

**常见错误**：

1. 用数据库最终值回测。
2. 不记录发布/到达时间。
3. 把“补发”回填到原日期。

---

## 5. 修订对回测的影响：一组数字

假设连续三期实际库存 104、100、96，初值/修订值如下：

| 期 | 初值 | 修订 | 实际 |
|---|---:|---:|---:|
| 1 | 100 | 108 | 104 |
| 2 | 102 | 104 | 100 |
| 3 | 104 | 100 | 96 |

信号 = 基准 − 实际（越小越看多）：

| 期 | 初值信号 | 修订信号 | 差异 |
|---|---:|---:|---:|
| 1 | −4 | +4 | 8 |
| 2 | +2 | +4 | 2 |
| 3 | +8 | +4 | 4 |

### 代码：信号差异



In [5]:

import numpy as np

initial = np.array([100, 102, 104])
revised = np.array([108, 104, 100])
actual = np.array([104, 100, 96])

sig_initial = initial - actual
sig_revised = revised - actual
print("初值信号", sig_initial, "修订信号", sig_revised)
print("绝对差异合计", int(np.abs(sig_initial - sig_revised).sum()))


初值信号 [-4  2  8] 修订信号 [4 4 4]
绝对差异合计 14



**读数**：三期里两期方向不同，绝对差异合计 14；若用修订值回测，信号的符号/强度都变了。

**常见错误**：

1. 只查一期修订。
2. 不比较“初值/修订”两套回测。
3. 不记录版本差异。

---



## 本章小结

### 一张表

| 概念 | 做法 | 用途 |
|---|---|---|
| PIT | 当时可见版本 | 防泄漏 |
| merge_asof | 按可见时间对齐 | 面板 |
| 版本 | 初值/修订/最终 | 追溯 |
| 延迟 | 发布/到达 | 真实可见 |

### 六句话

1. 回测用当时可见，不用最终值。
2. merge_asof 取“可见 ≤ 决策”的最近一条。
3. 发布 ≠ 到达。
4. 修订值会偷看未来。
5. 保留版本/时间戳。
6. 补发要当“晚到”。

---

## 本章习题

**1.** 初值 100、修订 108，09:06 应该看到哪个？

**2.** merge_asof 的 direction 用什么？

**3.** 为什么修订值不能用于当时信号？

**4.** 发布时刻和到达时刻有什么区别？

**5.** 补发数据应该怎么处理？

<details><summary>参考答案</summary>

**1.** 100（若 108 在 09:08 才可见）。

**2.** `backward`（取 ≤ 决策时刻的最近一条）。

**3.** 当时不知道修订；用了就是未来函数。

**4.** 发布是官方时间，到达是系统实际收到时间；网络/补发会晚。

**5.** 按“到达时间”入 PIT，不能回填到更早。

</details>

---

## 参考资料

- 第01、12章；Point-in-time 数据实践。
